# Project Forge — ComfyUI Remote Server

Este notebook transforma o Google Colab em um servidor ComfyUI gratuito com GPU.

**GPU:** T4 (12GB VRAM) — fornecida gratuitamente pelo Google.

---

## Como usar:
1. Menu `Runtime` → `Change runtime type` → `T4 GPU`
2. Execute cada célula em ordem
3. Quando a URL do túnel aparecer, copie (Ctrl+C) — o Forge detecta automaticamente

---

In [ ]:
# @title 1. Verificar GPU
import torch, psutil, platform, subprocess, os, json, time, threading, urllib.request

if torch.cuda.is_available():
    gpu = torch.cuda.get_device_properties(0)
    print(f"GPU: {gpu.name}")
    total_mem = getattr(gpu, 'total_memory', getattr(gpu, 'total_mem', 0))
    print(f"VRAM: {round(total_mem / 1024**3, 1)} GB")
else:
    print("❌ GPU não detectada. Vá em Runtime > Change runtime type > T4 GPU")
    raise SystemExit()

In [ ]:
# @title 2. Instalar dependências do sistema
!apt-get update -qq -y
!apt-get install -qq -y git wget unzip zip libgl1-mesa-glx libglib2.0-0 libsm6 libxext6 libxrender-dev libgomp1 > /dev/null 2>&1

# Instalar cloudflared (túnel)
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /usr/local/bin/cloudflared
!chmod +x /usr/local/bin/cloudflared

print("✅ Dependências instaladas")

In [ ]:
# @title 3. Baixar/Instalar ComfyUI
COMFY_DIR = "/content/ComfyUI"

if not os.path.exists(COMFY_DIR):
    print("Baixando ComfyUI...")
    !git clone --depth 1 https://github.com/comfyanonymous/ComfyUI.git "{COMFY_DIR}" 2>&1 | tail -1
else:
    print("ComfyUI já existe")

# Instalar requirements
print("Instalando dependências Python...")
!pip install -q -r "{COMFY_DIR}/requirements.txt" 2>&1 | tail -1

print("✅ ComfyUI pronto")

In [ ]:
# @title 4. Baixar modelo SD1.5
MODEL_NAME = "dreamshaper_8.safetensors"
MODEL_PATH = f"{COMFY_DIR}/models/checkpoints/{MODEL_NAME}"

if not os.path.exists(MODEL_PATH):
    print("Baixando Dreamshaper 8 (~2GB)...")
    urls = [
        "https://civitai.com/api/download/models/128713?type=Model&format=SafeTensor&size=pruned&fp=fp16",
        "https://huggingface.co/stable-diffusion-v1-5/stable-diffusion-v1-5/resolve/main/v1-5-pruned-emaonly.safetensors"
    ]
    ok = False
    for url in urls:
        try:
            !wget -q --show-progress -O "{MODEL_PATH}" "{url}"
            if os.path.exists(MODEL_PATH) and os.path.getsize(MODEL_PATH) > 1_000_000_000:
                ok = True
                break
            else:
                print("Falhou, tentando outra fonte...")
        except Exception as e:
            print(f"Erro na fonte {url}: {e}")
    if not ok:
        print("❌ Nao foi possivel baixar o modelo.")
        raise SystemExit()
    print("✅ Modelo baixado")
else:
    print("✅ Modelo já existe")

print(f"\nModelo: {MODEL_NAME}")
print(f"Tamanho: {round(os.path.getsize(MODEL_PATH) / 1024**3, 1)} GB")

In [ ]:
# @title 5. Baixar modelo de upscale (RealESRGAN)
import os
UPSCALE_DIR = f"{COMFY_DIR}/models/upscale_models"
UPSCALE_PATH = f"{UPSCALE_DIR}/RealESRGAN_x4plus.pth"
os.makedirs(UPSCALE_DIR, exist_ok=True)
if not os.path.exists(UPSCALE_PATH):
    !wget -q --show-progress -O "{UPSCALE_PATH}" "https://github.com/xinntao/Real-ESRGAN/releases/download/v0.1.0/RealESRGAN_x4plus.pth"
    print("Upscale model baixado!")
else:
    print("Upscale model ja existe.")


In [ ]:
# @title 6. (Opcional) Baixar VAE e LoRA extras
# Descomente se quiser modelos extras

# VAE melhorado
# vae_url = "https://huggingface.co/stabilityai/sd-vae-ft-mse-original/resolve/main/vae-ft-mse-840000-ema-pruned.safetensors"
# !wget -q --show-progress -O "{COMFY_DIR}/models/vae/vae-ft-mse.safetensors" "{vae_url}"

# ControlNet (opcional)
# !wget -q --show-progress ...


In [ ]:
# @title 7. Iniciar servidor ComfyUI
import subprocess, sys

server_port = 8188

log_file = open('/content/comfyui.log', 'w')

server = subprocess.Popen(
    [sys.executable, "main.py", f"--port={server_port}", "--listen=0.0.0.0"],
    cwd=COMFY_DIR,
    stdout=log_file,
    stderr=subprocess.STDOUT,
    text=True
)

print(f"✅ ComfyUI iniciado (PID: {server.pid})")
print("Aguardando servidor ficar pronto...")

# Aguardar servidor ficar acessível
import time as tmod
for i in range(120):
    try:
        req = urllib.request.Request("http://127.0.0.1:8188/system_stats")
        urllib.request.urlopen(req, timeout=2)
        print(f"✅ Servidor pronto após {i+1}s")
        break
    except:
        tmod.sleep(1)
else:
    print("❌ Servidor não iniciou. Verifique os logs em /content/comfyui.log")
    server.terminate()
    raise SystemExit()


In [ ]:
# @title 8. Tunel automatico + sincronizacao (deixe rodando)
import subprocess, threading, time, re, urllib.request, json

SYNC_URL = "https://jsonblob.com/api/jsonBlob/019fb890-6c80-7896-88e3-14ac5bf3ca7c"

TUNNEL_URL = [None]

def find_url(line):
    for pat in [r'https://[a-z0-9-]+\.lhr\.life',
                r'https://[a-z0-9-]+\.serveo\.net',
                r'https://[a-z0-9-]+\.trycloudflare\.com',
                r'https://[a-z0-9-]+\.loca\.lt']:
        m = re.search(pat, line)
        if m and "api.trycloudflare.com" not in m.group(0):
            return m.group(0)
    return None

def publish(url):
    try:
        data = json.dumps({"tunnel": url}).encode("utf-8")
        req = urllib.request.Request(
            SYNC_URL, data=data,
            headers={"Content-Type": "application/json"}, method="PUT")
        urllib.request.urlopen(req, timeout=15)
        print(f"[sync] URL publicada para o Forge: {url}")
    except Exception as e:
        print(f"[sync] erro ao publicar: {e}")

def drain(proc):
    try:
        for _ in proc.stdout:
            pass
    except Exception:
        pass

def try_tunnel(host):
    p = subprocess.Popen(
        ["ssh", "-o", "StrictHostKeyChecking=no",
         "-o", "ServerAliveInterval=30",
         "-R", "80:localhost:8188", host],
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    lines = []
    try:
        for line in p.stdout:
            lines.append(line.strip())
            if len(lines) > 20:
                lines.pop(0)
            u = find_url(line)
            if u:
                return u, p
    except Exception:
        pass
    return None, p

def run_tunnel():
    services = ["nokey@localhost.run", "serveo.net"]
    idx = 0
    while True:
        host = services[idx % len(services)]
        print(f"[tunel] tentando {host} ...")
        url, proc = try_tunnel(host)
        if url:
            TUNNEL_URL[0] = url
            print(f"[tunel] ATIVO: {url}")
            publish(url)
            threading.Thread(target=drain, args=(proc,), daemon=True).start()
        else:
            print("[tunel] nao conseguiu URL. Ultimas saidas:")
            try:
                for line in list(proc.stdout)[-8:]:
                    print("   " + line.strip())
            except Exception:
                pass
            try:
                proc.kill()
            except Exception:
                pass
            idx += 1
            time.sleep(3)
            continue
        try:
            proc.wait()
        except Exception:
            pass
        try:
            proc.kill()
        except Exception:
            pass
        TUNNEL_URL[0] = None
        print("[tunel] caiu. reconectando em 5s...")
        time.sleep(5)

if 'TUNNEL_AUTO_STARTED' in globals():
    print("Tunel automatico ja esta rodando nesta sessao.")
else:
    globals()['TUNNEL_AUTO_STARTED'] = True
    threading.Thread(target=run_tunnel, daemon=True).start()
    print("Tunel automatico ATIVO. Deixe o Colab aberto.")


In [ ]:
# @title 9. Status do tunel (diagnostico)
import urllib.request, time
try:
    u = TUNNEL_URL[0] if 'TUNNEL_URL' in globals() else None
except Exception:
    u = None
print("Ultima URL conhecida:", u)
if u:
    try:
        r = urllib.request.urlopen(urllib.request.Request(f"{u}/system_stats"), timeout=10)
        print(f"ComfyUI respondeu: HTTP {r.status} (tunel OK)")
    except Exception as e:
        print(f"Tunel sem resposta: {e}")


In [ ]:
# @title 10. Instrucoes finais
print("""
========================================
   PROJECT FORGE - CONFIGURADO
========================================
   O tunel automatico esta rodando e publicando
   a URL para o Forge. Nao precisa colar nada!

   O Forge detecta a URL nova sozinho (a cada 8s)
   e mostra CONECTADO quando o servidor responde.

   ATENCAO:
   - Deixe o Colab aberto e rodando.
   - A sessao expira em ~12h. Quando expirar,
     rode todas as celulas de novo (Executar tudo).
""")


In [ ]:
# @title ⏳ Keep Alive (evita desconexão)
# Esta célula mantém o notebook ativo.
# Execute se for usar por mais de 30min.

import time, threading, requests

def ping_loop():
    while True:
        try:
            requests.get("https://www.google.com", timeout=10)
        except:
            pass
        time.sleep(60)

if tunnel_url:
    t = threading.Thread(target=ping_loop, daemon=True)
    t.start()
    print("✅ Keep alive ativo (ping a cada 60s)")
else:
    print("Configure o túnel primeiro")